# Compare training configurations

**Question: which configuration performs well across different training seeds?**

1. Describe configurations and seeds once in `configs/experiment.json`.
2. Run `scripts/experiment.py` (or `--prepare` while training is unavailable).
3. Select its output directory in `EXPERIMENT` below, then run all cells.

The experiment supplies the run list, datasets, seeds, and grouping. This notebook
only reads saved results. Use [evaluation.ipynb](evaluation.ipynb) to inspect one model.


## 1. Select an experiment

Set `EXPERIMENT` to the directory containing `experiment.json`. Leave it as `None`
only for the optional standalone `RUNS` comparison. Standalone reports have no
configuration/seed metadata and cannot be averaged into training configurations.


In [ ]:
from pathlib import Path
import sys
from html import escape
import matplotlib.pyplot as plt
from IPython.display import HTML, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.comparison import load_comparison, load_experiment, load_history

EXPERIMENT = None  # --- Select here an experiment directory (ROOT / "results/experiments/NAME") ---

# Optional: compare standalone reports when EXPERIMENT is None.
RUNS = ["RUN1", "RUN2"]  # --- Select here evaluation runs ---
VALIDATION_DATASET = ROOT / "dataset/references/chess-v1/splits/validation.txt"


## 2. Compare configurations across seeds

Scores show **mean ± sample standard deviation**; this is not a confidence interval.
A single seed has no estimated standard deviation. Missing seeds remain explicit.
Only a finished experiment with every planned seed and at least two seeds can
establish leading configurations. Default seeds and all settings come from the
experiment file, not this notebook. Exact mean F1 ties use mean worst-class recall.

Timing includes loading/startup and is descriptive. See the
[protocol](../docs/evaluation.md#comparison-protocol-v1).


In [ ]:
summary = load_experiment(EXPERIMENT) if EXPERIMENT is not None else None
if summary is not None:
    comparison = summary["runs"]
    print(f"Experiment: {EXPERIMENT} | Status: {summary['status']}")
    print("Planned seeds:", summary["expected_seeds"])
    if summary["selection_ready"]:
        print("Leading configurations on validation:", ", ".join(summary["leading_configurations"]))
    else:
        print("No configuration selection: experiment unfinished, missing seeds, or only one planned seed.")
else:
    comparison = (load_comparison([ROOT / "results" / name for name in RUNS],
                                  VALIDATION_DATASET) if RUNS else [])
    print("Standalone reports: no seed aggregation without an experiment plan.")

def table(headings, rows):
    header = "".join(f"<th>{escape(str(value))}</th>" for value in headings)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

def score(stat):
    if stat["mean"] is None:
        return "—"
    if stat["std"] is None:
        return f"{stat['mean']:.3f} (one seed)"
    return f"{stat['mean']:.3f} ± {stat['std']:.3f}"

if summary is not None:
    table(("Configuration", "Seeds completed", "Accuracy", "Macro F1", "Worst recall", "Train (s)", "Predict (s)", "Model (KiB)"),
          [(row["configuration"], f"{len(row['seeds'])}/{len(summary['expected_seeds'])}",
            score(row["accuracy"]), score(row["macro_f1"]), score(row["worst_recall"]),
            score(row["training_seconds"]), score(row["prediction_seconds"]),
            f"{row['model_bytes']['mean'] / 1024:.1f}" if row["model_bytes"]["mean"] is not None else "—")
           for row in summary["configurations"]])
    for row in summary["configurations"]:
        if row["missing_seeds"]:
            print(f"{row['configuration']}: missing seeds {row['missing_seeds']}")


In [ ]:
if summary is not None:
    groups = [row for row in summary["configurations"] if row["macro_f1"]["mean"] is not None]
    if groups:
        fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
        for index, row in enumerate(groups):
            stat = row["macro_f1"]
            ax.errorbar(index, stat["mean"], yerr=stat["std"] if stat["std"] is not None else None,
                        fmt="o" if row["complete"] else "x", capsize=5, color="tab:blue")
        ax.set_xticks(range(len(groups)), [row["configuration"] + ("" if row["complete"] else " (partial)") for row in groups])
        ax.set_xlim(-0.5, len(groups) - 0.5)
        ax.set_ylim(-0.05, 1.05)
        ax.set_ylabel("Validation macro F1")
        ax.set_title("Mean and sample standard deviation across training seeds")
        ax.spines[["top", "right"]].set_visible(False)
        plt.show()


## 3. Inspect individual runs

These results explain variability and class-specific weaknesses. Do not select a
configuration using its luckiest seed. Open a report directory in `evaluation.ipynb`
for its confusion matrix.


In [ ]:
if comparison:
    table(("Run", "Accuracy", "Macro F1", "Worst recall", "Elapsed (s)"),
          [(report["run"], f"{report['metrics']['accuracy']:.2%}",
            f"{report['metrics']['macro_f1']:.3f}", f"{report['worst_recall']:.3f}",
            f"{report['metrics']['prediction_seconds']:.2f}") for report in comparison])
    print("Validation SHA-256:", comparison[0]["metrics"]["artifacts"]["dataset"]["sha256"])
else:
    print("No completed evaluations yet. Prepare/train an experiment, or select standalone validation RUNS.")


In [ ]:
if comparison:
    labels = comparison[0]["metrics"]["confusion_matrix"]["labels"]
    fig, axes = plt.subplots(1, 2, figsize=(12, max(4, 0.45 * len(comparison) + 2)),
                             layout="constrained")
    names = [report["run"] for report in comparison]
    width = 0.24
    for ax, key in zip(axes, ("f1", "recall")):
        for offset, label in enumerate(labels):
            positions = [i + (offset - 1) * width for i in range(len(comparison))]
            values = [report["metrics"]["per_class"][label][key] for report in comparison]
            bars = ax.barh(positions, values, width, label=label)
            ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
        ax.set_yticks(range(len(names)), names)
        ax.invert_yaxis()
        ax.set_xlim(0, 1.15)
        ax.set_xlabel("Score")
        ax.set_title("F1 by class" if key == "f1" else "Recall by class")
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, 1.18), ncols=3)
    plt.show()


## 4. Read training curves when available

A growing train/validation loss gap may indicate overfitting. Curves require the
backend's optional `history.csv`; missing curves are reported, never fabricated.


In [ ]:
if summary is not None:
    histories = [(report["run"], load_history(Path(report["directory"]).parent / "history.csv"))
                 for report in comparison]
    available = [(name, rows) for name, rows in histories if rows]
    if not available:
        print("No training curves yet. The backend may emit history.csv using the documented format.")
    for name, rows in available:
        fig, ax = plt.subplots(figsize=(7, 3), layout="constrained")
        ax.plot([row["epoch"] for row in rows], [row["train_loss"] for row in rows], label="Train")
        ax.plot([row["epoch"] for row in rows], [row["validation_loss"] for row in rows], label="Validation")
        ax.set(xlabel="Epoch", ylabel="Loss", title=name)
        ax.legend()
        ax.spines[["top", "right"]].set_visible(False)
        plt.show()
    for name, rows in histories:
        if not rows:
            print(f"{name}: no training history")


## Conclusions

Record the quality/stability/cost tradeoff and any weak class. Freeze the selected
configuration and model-selection rule before the reserved test evaluation.
The runner does not export a final model or close issue #8.
